# Wasserstein variable projection: understanding classification models    


In [1]:
import numpy as np                     
import sklearn as sk                      
import matplotlib.pyplot as plt      
import pandas as pd                    
from tqdm import tqdm                  
from sklearn import preprocessing
from sklearn.model_selection import train_test_split


In [2]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes import GaussianNB as NB

### coisas 

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2)

In [ ]:
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 

In [ ]:
X.dtypes

### 1: Load and pre-process Adult Census data

Let's first give ourself a dataset which we will use on this tutorial, the preprocessing comes mainly from there :
https://github.com/XAI-ANITI/StoryOfBias/blob/master/StoryOfBias.ipynb

You can download the dataset itself here : https://www.kaggle.com/datasets/wenruliu/adult-income-datase

This dataset has personal information such as age, number of years spent studying, and martial status associated with the yearly income; more precisely, we will look at if those persons have a yearly income higher than $50,000.

#### 1.1 Data loading

In [3]:
from LEFkit.data import LoadAndTreatAdultCensus

data_ohe = LoadAndTreatAdultCensus.get_treated_dataframe(verbose=False)

X_raw_col_names=data_ohe.columns

In [4]:
X = data_ohe.drop('Target', axis = 1)
Y = data_ohe['Target']

In [ ]:
X.shape, Y.shape

In [5]:
X_train, X_test , Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [ ]:
X_train_scaled = np.asarray(sk.preprocessing.StandardScaler().fit(X_train).transform(X_train))
X_test_scaled  = np.asarray(sk.preprocessing.StandardScaler().fit(X_test).transform(X_test))

### 2: Train different classifiers <a name="a-pred"></a>

The dataset Adult Income has many features in different scales, such as 'Age' and 'Capital Gain'. Some algorithms, such as Logistic Regression, are quite sensitive to different scales, since they are based in computations related to norms. In that cases, the classifier may be errouneously biased by some covariates with larger range. Because of that, it is desirable to standardize the features before training. One issue arises in this context: standardize after affine transformations makes no difference in our approach here. Specifically, the Wasserstein projection of a dataset is just a linear translation. If then it is standardized, the standardized version of the projected one is the same of the standardized version of the original dataset. It happens just because of the linearity of expectation. Then, if we standardize our data before training, our analysis would be lost. So we are going to examine only algorithms that are not sensitive to scaling.

#### 2.1 Logistic regression <a name="a-logi"></a>

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_LR = LogisticRegression(solver='lbfgs',max_iter=500)
clf_LR.fit(X_train_scaled,Y_train)
pred_logi=1.*clf_LR.predict_proba(X_test_scaled)[:,1]

#### no scaling:

In [ ]:
clf_LR.fit(X_train,Y_train)
pred_logi_noscale=1.*clf_LR.predict_proba(X_test)[:,1]

#### 2.2 Decision Tree <a name="a-DT"></a>

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf_DT=DecisionTreeClassifier(max_depth=5)
clf_DT.fit(X_train_scaled,Y_train)
pred_DT = clf_DT.predict_proba(X_test_scaled)[:,1]###returns predicted probs for each classes, im choosing the second one

In [ ]:
plt.plot(X_train.columns,clf_DT.feature_importances_)
plt.xticks(rotation=80)
plt.show()

##### no scaling:

In [ ]:
clf_DT.fit(X_train,Y_train)
pred_DT_noscale = clf_DT.predict_proba(X_test)[:,1]

#### 2.3 Gradient boosting <a name="a-DT"></a>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

clf_GB=GradientBoostingClassifier()
#clf_GB.fit(X_train_scaled,Y_train)
#pred_GB = clf_GB.predict_proba(X_test_scaled)[:,1]



##### no scaling:

In [ ]:
clf_GB.fit(X_train,Y_train)
pred_GB_noscale = clf_GB.predict_proba(X_test)[:,1]

In [ ]:
clf_GB.predict_proba(X_test).shape

#### 2.4 LDA

In [ ]:
sklearn_lda = LDA()
lda = sklearn_lda.fit(X_train, Y_train)
X_labels = lda.predict(X_train)
X_prob = lda.predict_proba(X_train)

X_test_labels_LDA = lda.predict(X_test) ###retorna a classe
X_test_prob_LDA = lda.predict_proba(X_test) ###retorna as probabilidades de estar em cada classe
 

#### 2.5 QDA

In [ ]:
sklearn_qda = QDA(store_covariance=True) # criando um objeto QDA
qda = sklearn_qda.fit(X_train, Y_train)
X_labels = qda.predict(X_train)
X_prob = qda.predict_proba(X_train)

X_test_labels_QDA=qda.predict(X_test)
X_test_prob_QDA = qda.predict_proba(X_test) 


#### 2.6 Naive Bayes

In [ ]:
NB_class = NB()
NB_class.fit(X_train, Y_train)
X_test_labels_NB=NB_class.predict(X_test)
X_test_prob_NB = NB_class.predict_proba(X_test) 

#### 2.7 Prediction accuracy <a name="a-acc"></a>

In [ ]:
Y_pred_logi=1*(pred_logi>0.5)
Y_pred_DT=1*(pred_DT>0.5)
Y_pred_GB=1*(pred_GB>0.5)

print('Accuracy WITH SCALING:')
print('Logistic regression accuracy is ', 1-np.mean(np.abs(Y_pred_logi-Y_test)))
print('Decision tree accuracy is ', 1-np.mean(np.abs(Y_pred_DT-Y_test)))
print('Gradient Boosting accuracy is ', 1-np.mean(np.abs(Y_pred_GB-Y_test)))

In [ ]:
Y_pred_logi=1*(pred_logi_noscale>0.5)
Y_pred_DT=1*(pred_DT_noscale>0.5)
Y_pred_GB=1*(pred_GB_noscale>0.5)

print('Accuracy WITHOUT SCALING:')
print('Logistic regression accuracy is ', 1-np.mean(np.abs(Y_pred_logi-Y_test)))
print('Decision tree accuracy is ', 1-np.mean(np.abs(Y_pred_DT-Y_test)))
print('Gradient Boosting accuracy is ', 1-np.mean(np.abs(Y_pred_GB-Y_test)))

In [ ]:
print("Accuracy LDA is ", np.mean(Y_test==X_test_labels_LDA))
print("Accuracy QDA is ", np.mean(Y_test==X_test_labels_QDA))
print("Accuracy NB is ", np.mean(Y_test==X_test_labels_NB))



Since Logistic Regression is sensitive to scaling and the accuracy of QDA is very low, we will use only Decision Tree, Gradient Boosting, LDA and Naive Bayes algorithms. All of them are going to be trained with the NONSCALED dataset. Since the basic idea of tree classifiers and regressors is to find the best cut, tree-based algorithms are blind to feature scaling. The generative models for classification, Linear Discriminant Analysis, Quadratic Discriminant Analysis and Naive Bayes, are based in estimating the posterior probability of belonging to a class. This approach is also blind to feature scaling.

### 3. Study a classifier stressing feature mean

In [ ]:
from classification_functions import *

#### 3.1 Mean influence in portin of predicted 1's

##### 3.1.1 mean influence for all four models

In order to understand the influence of each covariate in the classification, we are going to follow three basic steps: train all four models in the same dataset X_train, determine the projections parametrized by $\tau$ of the test dataset X_test, and then compute the proportion of predicted 1's for these projected datasets.

In [ ]:
plot_mean_pp1(X_train, X_test, Y_train, 21, 0.05, 'Age')

As we can see by the figure, the feature 'Age' has a beneficial impact in positive classifications ($\geq$ 50k income) up to an age close to 50. On the other hand, the impact of 'Education-Num' is clear: as higher 'Education-Num' is, the chance of having an income greater than 50k is bigger.

In [ ]:
plot_mean_pp1(X_train, X_test, Y_train, 21, 0.05, 'Education-Num')

In [ ]:
plot_mean_pp1(X_train, X_test, Y_train, 21, 0.05, 'Hours per week')

##### 3.1.2 Mean influence for one model

In [ ]:
plot_mean_pp1_onemodel(X_train, X_test, Y_train, 21, 0.05, 'Age', 'DT')

#### 3.2 Multiple mean stresses

In these 

In [ ]:
plot_multiplemean(X_train, X_test, Y_train, Y_test, 30, 0.05)

#### 3.3 Stressing two means simultaneously

In [ ]:
plot_twomeans(X_train, X_test, Y_train, 'Hours per week', 'Education-Num', 21,  0.05, 'GB')

### 4. Disparate Impact

In [ ]:
from LEFkit.bias_measure.bias_measure_fcts import *

In [ ]:
X_col_names = list(X_train.columns)
X_col_names.index('Gender')

In [ ]:
X_train['Child'].values

In [ ]:
X=X_test.values
S = X[:,X_col_names.index('Gender')].ravel()

In [ ]:
X.shape, S.shape, Y_pred_GB.shape

In [ ]:
Cpt_DI(S,Y_pred_GB,w=1 ,alpha=0.05, boxplot=False, wedge=False)

In [ ]:
#####generating new datasets and predictions for all models

In [ ]:
X_train['Capital Loss'].value_counts()

In [ ]:
x_train = X_train
x_test = X_test
y_train = Y_train 
n_tau = 21
alpha = 0.05
col_name = 'Capital Loss'

In [ ]:
###compute lambdas
list_lbd, list_t = comp_lambda(x_test, col_name, n_tau, alpha)
    
###compute new dataframes
dfs = []
for i in range(len(list_lbd)):
    data = new_dataset(x_test, col_name, list_lbd[i])
    dfs.append(data)

###instantiate the model
clf_DT=DecisionTreeClassifier(max_depth=5)
clf_DT.fit(x_train, y_train)

##compute portions of 1
dict_GB = {}
for i in range(len(dfs)):
    pred_GB = clf_DT.predict_proba(dfs[i])[:,1]
    Y_pred_GB=1*(pred_GB>0.5)
    dict_GB[list_t[i]] = Y_pred_GB
    print(len(dict_GB.keys()))

In [ ]:
list1 = [11, 15, 18, 20]
for i in list1:
    pred_GB = clf_GB.predict_proba(dfs[i])[:,1]
    Y_pred_GB=1*(pred_GB>0.5)
    dict_GB[list_t[i]] = Y_pred_GB

In [ ]:
len(dict_GB.keys())

In [ ]:
dfs2 = [dfs[11], dfs[15], dfs[18], dfs[20]]
dict_GB = {}
for i in range(len(dfs2)):
    pred_GB = clf_DT.predict_proba(dfs2[i])[:,1]
    Y_pred_GB=1*(pred_GB>0.5)
    dict_GB[list_t[i]] = Y_pred_GB
    #print(dict_GB.keys())

In [ ]:
list_t

In [ ]:
len(dict_GB.keys())

In [ ]:
len(dict_GB.keys())

In [ ]:
dis = []
yerr = []
sizes = []
for key in dict_GB.keys():
    y = dict_GB[key]
    di = Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[0]
    errs =  Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[1]
    dis.append(di)
    yerr.append(errs)

In [ ]:
len(list_t), len(dis), len(dict_GB), len(dfs)

In [ ]:
list_t[10]

In [ ]:
np.mean(x_test['Age'])

In [ ]:
plt.style.use('default')

In [ ]:
plt.errorbar(list_t, dis, yerr = sizes, color = 'k', capsize=2, marker="s", markersize=3.5, mfc="k", mec="k", alpha = 0.65)
plt.plot(list_t[10], dis[10], marker = '*',markersize=10, color = 'r')
plt.xlabel(f'mean {col_name}')
plt.ylabel("DI")
plt.ylim(0.2,0.5)
plt.show()

In [ ]:
a = np.arange(3)
b = np.arange(3)-1
c = zip(a,b)
print(c)

In [ ]:
def plot_di4(x_train, x_test, y_train, n_tau, alpha):
    cols = ['Age', 'Education-Num', 'Capital Gain', 'Hours per week']
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(12, 4))
    axs = [ax1, ax2, ax3, ax4]
    for col_name, ax in zip(cols, axs):
        list_t, dis, sizes = plot_di_one(x_train, x_test, y_train, n_tau, alpha, col_name)
        ax.errorbar(list_t, dis, yerr = sizes, color = 'k', capsize=2, marker="s", markersize=3.5, mfc="k", mec="k", alpha = 0.65)
        ax.plot(list_t[10], dis[10], marker = '*',markersize=10, color = 'r')
        ax.set_xlabel(f'Average {col_name}')
        ax.set_ylim(0.2,0.5)
    ax1.set_ylabel("DI")
    fig.tight_layout()
    plt.show()


In [ ]:
plot_di4(x_train, x_test, y_train, n_tau, alpha)

In [ ]:
def plot_di_one(x_train, x_test, y_train, n_tau, alpha, col_name):
    ###compute lambdas
    list_lbd, list_t = comp_lambda(x_test, col_name, n_tau, alpha)
    
    ###compute new dataframes
    dfs = []
    for i in range(len(list_lbd)):
        data = new_dataset(x_test, col_name, list_lbd[i])
        dfs.append(data)

    ###instantiate the model
    clf_GB = GradientBoostingClassifier()
    clf_GB.fit(x_train, y_train)

    ##compute portions of 1
    dict_GB = {}
    for i in range(len(dfs)):
        pred_GB = clf_GB.predict_proba(dfs[i])[:,1]
        Y_pred_GB=1*(pred_GB>0.5)
        dict_GB[list_t[i]] = Y_pred_GB
    
    dis = []
    yerr = []
    for key in dict_GB.keys():
        y = dict_GB[key]
        di = Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[0]
        errs =  Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[1]
        dis.append(di)
        yerr.append(errs)
    
    sizes = []
    for i in range(len(yerr)):
        err = yerr[i][1] - dis[i]
        sizes.append(err)

    return list_t, dis, sizes

In [ ]:
def plot_di(x_train, x_test, y_train, n_tau, alpha, col_name):
    plt.style.use("default")
    ###compute lambdas
    list_lbd, list_t = comp_lambda(x_test, col_name, n_tau, alpha)
    
    ###compute new dataframes
    dfs = []
    for i in range(len(list_lbd)):
        data = new_dataset(x_test, col_name, list_lbd[i])
        dfs.append(data)

    ###instantiate the model
    clf_GB = GradientBoostingClassifier()
    clf_GB.fit(x_train, y_train)

    ##compute portions of 1
    dict_GB = {}
    for i in range(len(dfs)):
        pred_GB = clf_GB.predict_proba(dfs[i])[:,1]
        Y_pred_GB=1*(pred_GB>0.5)
        dict_GB[list_t[i]] = Y_pred_GB
    
    dis = []
    yerr = []
    for key in dict_GB.keys():
        y = dict_GB[key]
        di = Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[0]
        errs =  Cpt_DI(S, y,w=1 ,alpha=0.05, boxplot=False, wedge=False)[1]
        dis.append(di)
        yerr.append(errs)
    
    sizes = []
    for i in range(len(yerr)):
        err = yerr[i][1] - dis[i]
        sizes.append(err)

    plt.errorbar(list_t, dis, yerr = sizes, color = 'k', capsize=2, marker="s", markersize=3.5, mfc="k", mec="k", alpha = 0.65)
    plt.plot(list_t[10], dis[10], marker = '*',markersize=10, color = 'r')
    plt.xlabel(f'mean {col_name}')
    plt.ylabel("DI")
    #plt.ylim(0.2,0.5)
    plt.show()
 

In [ ]:
cols = ['Age', 'Education-Num', 'Capital Gain', 'Hours per week']
for colname in cols:
    plot_di(x_train, x_test, y_train, n_tau, alpha, colname)

In [ ]:
def plot_di_mean(x_train, x_test, y_train, n_tau, alpha, col_name, model):
    plt.style.use("seaborn")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for col_name in 
    ###compute lambdas
    list_lbd, list_t = comp_lambda(x_test, col_name, n_tau, alpha)

    ###compute new dataframes
    dfs = []
    for i in range(len(list_lbd)):
        data = new_dataset(x_test, col_name, list_lbd[i])
        dfs.append(data)

    ###instantiate the model
    clf_GB = GradientBoostingClassifier()
    clf_GB.fit(x_train, y_train)

    ##compute portions of 1
    dict_GB = {}
    for i in range(len(dfs)):
        pred_GB = clf_GB.predict_proba(dfs[i])[:,1]
        Y_pred_GB=1*(pred_GB>0.5)
        dict_GB[list_t[i]] = Y_pred_GB


In [ ]:

    #n = Y_pred_GB.shape[0]
    #portion = np.sum(Y_pred_GB)/n
    #portions_GB.append(portion) 
    
##instantiate the model
clf_DT=DecisionTreeClassifier(max_depth=5)
clf_DT.fit(x_train, y_train)

##compute portions of 1
portions_DT = []
for i in range(len(dfs)):
    pred_DT = clf_DT.predict_proba(dfs[i])[:,1]
    Y_pred_DT=1*(pred_DT>0.5)
    n = Y_pred_DT.shape[0]
    portion = np.sum(Y_pred_DT)/n
    portions_DT.append(portion)
   
##instantiate the model
sklearn_lda = LDA()
lda = sklearn_lda.fit(x_train, y_train)
        
##compute portions of 1
portions_LDA = []
for i in range(len(dfs)):
    pred_lda = lda.predict_proba(dfs[i])[:,1]
    Y_pred_lda=1*(pred_lda>0.5)
    n = Y_pred_lda.shape[0]
    portion = np.sum(Y_pred_lda)/n
    portions_LDA.append(portion)


##instantiate the model
clf_NB = NB()
clf_NB.fit(x_train, y_train)
        
##compute portions of 1
portions_NB = []
for i in range(len(dfs)):
    X_test_prob_NB = clf_NB.predict_proba(dfs[i])[:,1]
    Y_pred_NB=1*(X_test_prob_NB>0.5)
    n = Y_pred_NB.shape[0]
    portion = np.sum(Y_pred_NB)/n
    portions_NB.append(portion)

### 5.0 feature importance xgboost

### 5. SHAP

In [ ]:
import shap
import xgboost
# print the JS visualization code to the notebook
shap.initjs()

In [ ]:
Y_train.shape

In [ ]:
d_train = xgboost.DMatrix(X_train, label=Y_train)
d_test = xgboost.DMatrix(X_test, label=Y_test)
params = {
    "eta": 0.01,
    "objective": "binary:logistic",
    "subsample": 0.5,
    "base_score": np.mean(Y_train),
    "eval_metric": "logloss",
}
model = xgboost.train(
    params,
    d_train,
    5000,
    evals=[(d_test, "test")],
    verbose_eval=100,
    early_stopping_rounds=20,
)


In [ ]:
xgboost.plot_importance(model, max_num_features =10, ylabel = None, xlabel = None) #is the number of times a feature appears in a tree
plt.title("Feature importance weight")
#plt.savefig("xgboost_importance.png", dpi=300, bbox_inches="tight")
#plt.close()
plt.show()

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.plots.beeswarm(explainer(X_test))
#plt.savefig("shap_summary2.png", dpi=300, bbox_inches="tight")
#plt.close()

In [ ]:
values = explainer(X_test).values

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
shap.plots.beeswarm(explainer(X_test), ax=ax1, plot_size=None, show = False)
ax1.set_title("SHAP values")
ax1.set_xlabel(xlabel=None)

xgboost.plot_importance(model, ax = ax2, max_num_features =10, ylabel = None, xlabel=None) #is the number of times a feature appears in a tree
ax2.set_title("Feature importance score")
plt.tight_layout()
plt.show()

In [ ]:
Y.shape


In [ ]:
X.shape

In [ ]:
# this takes a minute or two since we are explaining over 30 thousand samples in a model with over a thousand trees
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

Visualizing the first predicion:

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[0, :], X_display.iloc[0, :])

In [ ]:
##visualizing the first 1000 predictions
shap.force_plot(explainer.expected_value, shap_values[:1000, :], X_display.iloc[:1000, :])

In [ ]:
shap.summary_plot(shap_values, X_display, plot_type="bar")

In [ ]:
shap.summary_plot(shap_values, X)

In [ ]:
model

In [ ]:
X_test.shape

### training shap on gradient boosting

In [ ]:
#model = clf_GB
explainer = shap.TreeExplainer(clf_GB)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
shap.plots.beeswarm(explainer(X_test),show=False)
plt.savefig("shap_summary2.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[12, :], X_test.iloc[12, :])

In [ ]:
shap.force_plot(explainer.expected_value, shap_values[:1000, :], X_test.iloc[:1000, :])

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
X_train.columns

In [ ]:
shap.dependence_plot('Capital Loss', shap_values, X_test, display_features=X_test)

In [ ]:
shap.dependence_plot('Capital Gain', shap_values, X_test, display_features=X_test)

In [ ]:
X_test['Education-Num'].min(), X_test['Education-Num'].max()

In [ ]:
shap.dependence_plot('Education-Num', shap_values, X_test, display_features=X_test)

In [ ]:
shap.dependence_plot('Age', shap_values, X_test, display_features=X_test,show=False)
plt.savefig("dep_age.png", dpi=300, bbox_inches="tight")
plt.close()


In [ ]:
shap.plots.waterfall(explainer(X_test)[75], show = False)
plt.savefig("waterfall_individual2.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
1+2

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

shap.plots.waterfall(explainer(X_test)[0], show = False)
ax1.set_title("waterfall_individual")

shap.dependence_plot('Age', shap_values, X_test, display_features=X_test, show = False)
ax2.set_title("dep_age")

shap.plots.beeswarm(explainer(X_test), show = False)
ax3.set_title("beeswarm")

plt.tight_layout()
plt.show()

In [ ]:
X_test.shape

In [ ]:
X_test['Age'].min(), X_test['Age'].max()

In [ ]:
shap.dependence_plot('Hours per week', shap_values, X_test, display_features=X_test)

In [ ]:
shap.summary_plot(shap_values)

In [ ]:
shap.summary_plot(shap_values, plot_type='violin')

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.bar(shap_values[0])

In [ ]:
model.plot_importance(model)
plt.title("xgboost.plot_importance(model)")
pl.show()

### 6.entropic

In [ ]:
import ethik

explainer = ethik.ClassificationExplainer()

In [ ]:
y_pred = model.predict(d_test)

# We use a named pandas series to make plot labels more explicit
y_pred = pd.Series(y_pred, name='>$50k')

In [ ]:
explainer.plot_influence(
    X_test=X_test['Age'],
    y_pred=y_pred
)

In [ ]:
explainer.plot_influence(
    X_test=X_test['Age'],#, 'Education-Num', 'Capital Gain',
       #'Capital Loss', 'Hours per week']],
    y_pred=y_pred,
    colors={
        'Age': 'red',
        #'hours-per-week': 'green',
        #'Education-Num': 'blue',
        #'Capital Gain': 'orange',
        #'Capital Loss': 'purple'
    }
)

### gems explainer

In [9]:
from GEMS3_classification_explainer import plot_mean_influence_on_pred

ImportError: cannot import name 'plot_mean_influence_on_pred' from 'GEMS3_classification_explainer' (c:\Users\user\wass_projec\GEMS3_classification_explainer.py)

In [7]:
plot_mean_influence_on_pred(X_test, 'Age')

NameError: name 'plot_mean_influence_on_pred' is not defined